In [1]:
import os

# 1. Clone your GitHub repository and move into it
!git clone https://github.com/hbdz936/CodeDebtAI.git
%cd CodeDebtAI

# 2. Install your required dependencies
!pip install -r multi-agent/requirements.txt

Cloning into 'CodeDebtAI'...
remote: Enumerating objects: 245, done.
remote: Counting objects: 100% (245/245), done.
remote: Compressing objects: 100% (173/173), done.
remote: Total 245 (delta 82), reused 193 (delta 53), pack-reused 0 (from 0)
Receiving objects: 100% (245/245), 304.66 KiB | 5.75 MiB/s, done.
Resolving deltas: 100% (82/82), done.
/content/CodeDebtAI
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.4/149.4 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.5/106.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.9/434.9 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 7.6 MB/s eta 0:00:00
  Attempting uninstall: python-dotenv
    Found existing installation: python-dotenv 1.2.2
    Uninstalling python-dotenv-1.2.2:
      Successfully uninstall

In [8]:
import getpass
print("Please enter your Groq API Key to run the evaluation:")
os.environ["GROQ_API_KEY"] = getpass.getpass()

Please enter your Groq API Key to run the evaluation:
··········


In [9]:
import sys
import os
import time
import ast

# Add the multi-agent directory to the Colab path so we can import your agent
sys.path.append(os.path.abspath('multi-agent'))
from fixing.code_fixing_agent import generate_fix

print("✅ Agent successfully imported from your GitHub Repo!")

✅ Agent successfully imported from your GitHub Repo!


In [10]:
# Define our test dataset of "messy" code snippets that simulate technical debt
test_cases = [
    {
        "name": "Nested Ifs (High Cyclomatic Complexity)",
        "issue": "High cyclomatic complexity due to deeply nested if statements.",
        "code": "def check_user(user):\n    if user:\n        if user.is_active:\n            if user.has_permission:\n                return True\n    return False"
    },
    {
        "name": "Global Variables",
        "issue": "Use of global variables makes the function impure and hard to test.",
        "code": "count = 0\ndef increment():\n    global count\n    count += 1\n    return count"
    },
    {
        "name": "Bare Except Clause",
        "issue": "Using a bare except clause hides potential bugs and makes debugging difficult.",
        "code": "def divide(a, b):\n    try:\n        return a / b\n    except:\n        return 0"
    },
    {
        "name": "Inefficient For-Loop",
        "issue": "Using a standard loop to append to a list instead of a list comprehension.",
        "code": "def get_squares(numbers):\n    result = []\n    for n in numbers:\n        result.append(n * n)\n    return result"
    },
    {
        "name": "Unused Imports & Variables",
        "issue": "Contains unused imports and variables which clutter the code.",
        "code": "import math\nimport os\n\ndef calculate_area(radius):\n    unused_var = 10\n    return 3.14 * radius * radius"
    }
]

In [11]:
# Variables to track our metrics
total_time = 0
successful_parses = 0
failed_parses = 0
results = []

print("Starting Evaluation Pipeline...\n")

for i, test in enumerate(test_cases):
    print(f"Evaluating Case {i+1}: {test['name']}")
    start_time = time.time()

    # 1. Call the actual LLM agent from your project to generate a fix
    try:
        suggested_code = generate_fix(test["code"], test["issue"])
    except Exception as e:
        print(f"  ❌ API Error: {e}")
        continue

    end_time = time.time()
    latency = end_time - start_time
    total_time += latency

    # 2. Run AST Parsing Validation (The exact same check your graph pipeline uses)
    is_valid = False
    try:
        ast.parse(suggested_code)
        is_valid = True
        successful_parses += 1
        print(f"  ✅ Validation Passed (Valid Python Syntax)")
    except SyntaxError as e:
        failed_parses += 1
        print(f"  ❌ Validation Failed (Syntax Error): {e}")

    print(f"  ⏱️ Latency: {latency:.2f} seconds\n")

Starting Evaluation Pipeline...

Evaluating Case 1: Nested Ifs (High Cyclomatic Complexity)
  ✅ Validation Passed (Valid Python Syntax)
  ⏱️ Latency: 0.18 seconds

Evaluating Case 2: Global Variables
  ✅ Validation Passed (Valid Python Syntax)
  ⏱️ Latency: 0.28 seconds

Evaluating Case 3: Bare Except Clause
  ✅ Validation Passed (Valid Python Syntax)
  ⏱️ Latency: 2.25 seconds

Evaluating Case 4: Inefficient For-Loop
  ✅ Validation Passed (Valid Python Syntax)
  ⏱️ Latency: 1.23 seconds

Evaluating Case 5: Unused Imports & Variables
  ✅ Validation Passed (Valid Python Syntax)
  ⏱️ Latency: 20.25 seconds



In [12]:
# Calculate and Display Final Metrics
total_cases = len(test_cases)
success_rate = (successful_parses / total_cases) * 100 if total_cases > 0 else 0
avg_latency = total_time / total_cases if total_cases > 0 else 0

print("==========================================")
print("🏆 FINAL EVALUATION METRICS 🏆")
print("==========================================")
print(f"Total Files Analyzed   : {total_cases}")
print(f"Parsing Success Rate   : {success_rate:.1f}%")
print(f"Average LLM Latency    : {avg_latency:.2f} seconds")
print(f"Retry Exhaustion Risk  : {100 - success_rate:.1f}% (Code requiring retry)")
print("==========================================")

🏆 FINAL EVALUATION METRICS 🏆
Total Files Analyzed   : 5
Parsing Success Rate   : 100.0%
Average LLM Latency    : 4.84 seconds
Retry Exhaustion Risk  : 0.0% (Code requiring retry)
